In [1]:
!pip install tensorflowjs --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.19.0 which is incompatible.
google-cloud-bigquery 3.43.0 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have t

In [2]:
from google.colab import files
uploaded = files.upload()   # select fusion_lstm.weights.h5

Saving fusion_lstm.weights.h5 to fusion_lstm.weights.h5


In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models

WINDOW_SIZE = 15
N_FEATURES  = 10

def build_fusion_model():
    model = models.Sequential([
        layers.Input(shape=(WINDOW_SIZE, N_FEATURES)),
        layers.LSTM(32, return_sequences=False),
        layers.Dropout(0.3),
        layers.Dense(16, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid"),
    ])
    return model

model = build_fusion_model()
model.load_weights('fusion_lstm.weights.h5')
print("Weights loaded successfully — no deserialization involved, so no version sensitivity")
model.summary()

Weights loaded successfully — no deserialization involved, so no version sensitivity


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         5,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,049 (23.63 KB)

 Trainable params: 6,049 (23.63 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# HDF5, not the newer .keras format — avoids the Sequential-model
# weight-naming mismatch between Keras 3 and the tensorflowjs converter.
model.save('fusion_lstm_fixed.h5', save_format='h5')
print("Saved as HDF5")

Saved as HDF5


In [10]:
import os
print(os.path.exists('fusion_lstm_fixed.h5'))
print(os.path.getsize('fusion_lstm_fixed.h5'))

True
48304


In [11]:
!tensorflowjs_converter --input_format=keras fusion_lstm_fixed.h5 fusion_lstm_web

2026-08-19 10:08:42.267660: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787134122.323336    4962 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787134122.339882    4962 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787134122.382601    4962 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787134122.382670    4962 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787134122.382675    4962 computation_placer.cc:177] computation placer alr

In [12]:
import os
print(os.path.exists('fusion_lstm_web'))
if os.path.exists('fusion_lstm_web'):
    print(os.listdir('fusion_lstm_web'))

True
['model.json', 'group1-shard1of1.bin']


In [13]:
import json
with open('fusion_lstm_web/model.json') as f:
    data = json.load(f)
layers_list = data['modelTopology']['model_config']['config']['layers']
for l in layers_list:
    cfg = l['config']
    print(l['class_name'], cfg.get('name'), cfg.get('batchInputShape') or cfg.get('batch_shape'))

InputLayer input_layer [None, 15, 10]
LSTM lstm None
Dropout dropout None
Dense dense None
Dropout dropout_1 None
Dense dense_1 None


In [14]:
import shutil
shutil.make_archive('fusion_lstm_web', 'zip', 'fusion_lstm_web')

from google.colab import files
files.download('fusion_lstm_web.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>